# Embeddings

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath("../src"))

In [32]:
from embeddings.vectorizers import (
    build_char_ngrams,
    build_ngram_vectorizer,
    build_tfidf,
)

## 1.   vectorization

#### a.   Bag of Words (BoW)

In [35]:
from sklearn.feature_extraction.text import CountVectorizer

documents = [
    "the delivery was slow",
    "the product quality is great",
    "delivery delivery delivery was fast",
]

vectorizer = CountVectorizer()
vecteurs = vectorizer.fit_transform(documents)

print("Vocabulaire appris :", vectorizer.get_feature_names_out())
print()
print("Matrice de comptages (lignes = documents, colonnes = mots du vocabulaire) :")
print(vecteurs.toarray())

Vocabulaire appris : ['delivery' 'fast' 'great' 'is' 'product' 'quality' 'slow' 'the' 'was']

Matrice de comptages (lignes = documents, colonnes = mots du vocabulaire) :
[[1 0 0 0 0 0 1 1 1]
 [0 0 1 1 1 1 0 1 0]
 [3 1 0 0 0 0 0 0 1]]


#### b. TF-IDF

In [36]:
vectors, vectorizer = build_tfidf(documents)
print("Vocabulaire appris :", vectorizer.get_feature_names_out())
print()
print("Matrice TF-IDF (lignes = documents, colonnes = mots du vocabulaire) :")
print(vectors.toarray().round(3))

Vocabulaire appris : ['delivery' 'fast' 'great' 'is' 'product' 'quality' 'slow' 'the' 'was']

Matrice TF-IDF (lignes = documents, colonnes = mots du vocabulaire) :
[[0.46  0.    0.    0.    0.    0.    0.605 0.46  0.46 ]
 [0.    0.    0.467 0.467 0.467 0.467 0.    0.355 0.   ]
 [0.876 0.384 0.    0.    0.    0.    0.    0.    0.292]]


#### c.  n-grams words

In [ ]:
# affiche les n-grammes
print("N-grammes appris :", vectorizer.get_feature_names_out())
print()

print(
    "Matrice TF-IDF avec n-grammes (lignes = documents, "
    "colonnes = mots du vocabulaire) :"
)
vectors, vectorizer = build_ngram_vectorizer(
    documents, ngram_range=(1, 2), use_tfidf=True
)
print(vectors.toarray().T.round(3))

N-grammes appris : ['delivery' 'fast' 'great' 'is' 'product' 'quality' 'slow' 'the' 'was']

Matrice TF-IDF avec n-grammes (lignes = documents, colonnes = mots du vocabulaire) :
[[0.33  0.    0.649]
 [0.    0.    0.569]
 [0.33  0.    0.216]
 [0.    0.    0.284]
 [0.    0.341 0.   ]
 [0.    0.341 0.   ]
 [0.    0.341 0.   ]
 [0.    0.341 0.   ]
 [0.    0.341 0.   ]
 [0.    0.341 0.   ]
 [0.    0.341 0.   ]
 [0.434 0.    0.   ]
 [0.33  0.26  0.   ]
 [0.434 0.    0.   ]
 [0.    0.341 0.   ]
 [0.33  0.    0.216]
 [0.    0.    0.284]
 [0.434 0.    0.   ]]


#### c.  n-grams character


In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

mots = ["delivery", "delivry"]  # le second a une faute de frappe

# Vectorisation au niveau MOT (chaque mot entier = 1 unité)
vec_word = CountVectorizer(analyzer="word", token_pattern=r"(?u)\b\w+\b")
vec_word.fit(mots)
print("Vocabulaire mots :", vec_word.get_feature_names_out())
print("-> 'delivery' et 'delivry' sont 2 entrées TOTALEMENT différentes\n")

# Vectorisation au niveau CARACTERE, n-grams de 3
vec_char = CountVectorizer(analyzer="char", ngram_range=(3, 3))
vec_char.fit(mots)
vecteurs = vec_char.transform(mots).toarray()
vocab_char = vec_char.get_feature_names_out()

print("Vocabulaire caractères (3-grams) :", list(vocab_char))
print()

# Calcul du chevauchement (combien de 3-grams partagés)


v1, v2 = vecteurs[0], vecteurs[1]
chevauchement = np.minimum(v1, v2).sum()
total = min(v1.sum(), v2.sum())
print(f"3-grams partagés entre 'delivery' et 'delivry' : {chevauchement}/{total}")

Vocabulaire mots : ['delivery' 'delivry']
-> 'delivery' et 'delivry' sont 2 entrées TOTALEMENT différentes

Vocabulaire caractères (3-grams) : ['del', 'eli', 'ery', 'ive', 'ivr', 'liv', 'ver', 'vry']

3-grams partagés entre 'delivery' et 'delivry' : 3/5


In [39]:
# test de la fonction build_char_ngrams
documents = ["delivery", "delivry"]
vectors, vectorizer = build_char_ngrams(documents, ngram_range=(3, 3), use_tfidf=True)
print("Vocabulaire caractères (3-grams) :", vectorizer.get_feature_names_out())
print("Matrice TF-IDF avec 3-grams (lignes = documents, colonnes = 3-grams) :")
print(vectors.toarray().T.round(3))

Vocabulaire caractères (3-grams) : ['del' 'eli' 'ery' 'ive' 'ivr' 'liv' 'ver' 'vry']
Matrice TF-IDF avec 3-grams (lignes = documents, colonnes = 3-grams) :
[[0.335 0.379]
 [0.335 0.379]
 [0.47  0.   ]
 [0.47  0.   ]
 [0.    0.533]
 [0.335 0.379]
 [0.47  0.   ]
 [0.    0.533]]


## 2. Word2vec  

In [41]:
from embeddings.word2vec import get_word_vector, most_similar_words, train_word2vec

In [77]:
phrases_base = [
    "the delivery was fast".split(),
    "the delivery was slow".split(),
    "the shipping was fast".split(),
    "the shipping was slow".split(),
    "the delivery arrived quickly".split(),
    "the shipping arrived quickly".split(),
    "the product quality was great".split(),
    "the product quality was poor".split(),
    "customer service was helpful".split(),
    "customer service was rude".split(),
]

model_small = train_word2vec(phrases_base, vector_size=50, window=3, epochs=200)

print("Voisins de 'delivery' (corpus minuscule) :")
for mot, score in most_similar_words(model_small, "delivery", topn=30):
    print(f"  {mot:15} similarite = {score:.3f}")

print("\nVecteurs de mots :")

for mot, score in most_similar_words(model_small, "delivery", topn=2):
    print(f"  {mot:12} vector= {get_word_vector(model_small, mot).round(3)}")

Voisins de 'delivery' (corpus minuscule) :
  rude            similarite = 0.399
  slow            similarite = 0.296
  the             similarite = 0.265
  shipping        similarite = 0.250
  poor            similarite = 0.223
  product         similarite = 0.103
  arrived         similarite = 0.086
  was             similarite = 0.024
  quality         similarite = -0.008
  great           similarite = -0.029
  service         similarite = -0.045
  customer        similarite = -0.060
  helpful         similarite = -0.061
  fast            similarite = -0.180
  quickly         similarite = -0.262

Vecteurs de mots :
  rude         vector= [ 0.004  0.012  0.009 -0.015  0.014  0.001 -0.008  0.001  0.017  0.014
  0.014  0.     0.012 -0.003  0.008  0.006 -0.01  -0.011 -0.018 -0.014
 -0.003 -0.002 -0.014 -0.003  0.006 -0.011 -0.006 -0.005 -0.    -0.006
  0.01  -0.007 -0.004 -0.004 -0.013  0.008 -0.011 -0.008  0.012  0.019
  0.007  0.017  0.015 -0.001  0.011 -0.007 -0.001  0.002  0.019  0.0

In [83]:
# --- BLOC 3 : corpus repete (x50) pour simuler plus de donnees ---
phrases = phrases_base * 50

model = train_word2vec(phrases, vector_size=5, window=3, epochs=100, seed=42)

print("Voisins de 'delivery' (corpus x50) :")
for mot, score in most_similar_words(model, "delivery", topn=5):
    print(f"  {mot:12} similarite = {score:.3f}")

for mot, score in most_similar_words(model, "delivery", topn=2):
    print(f"  {mot:12} vector= {get_word_vector(model, mot).round(3)}")

print(f"  {get_word_vector(model, 'delivery').round(3)}")

cosine_similarity = np.dot(
    get_word_vector(model, "delivery"), get_word_vector(model, "shipping")
) / (
    np.linalg.norm(get_word_vector(model, "delivery"))
    * np.linalg.norm(get_word_vector(model, "shipping"))
)
print(f"Cosine similarity entre 'delivery' et 'shipping' : {cosine_similarity:.3f}")

Voisins de 'delivery' (corpus x50) :
  shipping     similarite = 0.998
  quickly      similarite = 0.955
  arrived      similarite = 0.790
  fast         similarite = 0.784
  slow         similarite = 0.769
  shipping     vector= [ 0.011  0.82   1.256  0.221 -0.301]
  quickly      vector= [-0.47   1.012  1.85   0.3   -0.715]
  [ 0.083  0.724  1.192  0.239 -0.269]
Cosine similarity entre 'delivery' et 'shipping' : 0.998


In [71]:
# --- BLOC 4 : CBOW (sg=0) vs Skip-gram (sg=1) ---
phrases_base2 = phrases_base + ["the packaging was damaged".split()]
phrases2 = phrases_base2 * 50

model_cbow = train_word2vec(
    phrases2, vector_size=50, window=3, sg=0, epochs=100, seed=42
)
model_skipgram = train_word2vec(
    phrases2, vector_size=50, window=3, sg=1, epochs=100, seed=42
)

print(
    "CBOW - voisins de 'delivery':", most_similar_words(model_cbow, "delivery", topn=3)
)
print(
    "Skip-gram - voisins de 'delivery':",
    most_similar_words(model_skipgram, "delivery", topn=3),
)
print()
print(
    "CBOW - voisins de 'packaging':",
    most_similar_words(model_cbow, "packaging", topn=3),
)
print(
    "Skip-gram - voisins de 'packaging':",
    most_similar_words(model_skipgram, "packaging", topn=3),
)

CBOW - voisins de 'delivery': [('shipping', 0.9970637559890747), ('arrived', 0.89862459897995), ('quickly', 0.8766067624092102)]
Skip-gram - voisins de 'delivery': [('shipping', 0.9961312413215637), ('quickly', 0.8454437851905823), ('arrived', 0.8291676640510559)]

CBOW - voisins de 'packaging': [('damaged', 0.7437599301338196), ('was', 0.704200267791748), ('slow', 0.6906068325042725)]
Skip-gram - voisins de 'packaging': [('damaged', 0.6933912038803101), ('was', 0.6412721276283264), ('quality', 0.6330041289329529)]


In [72]:
# --- BLOC 5 : similarite entre mots relies vs mots sans rapport ---
print("Mots relies (contextes similaires) :")
print("  delivery <-> shipping :", model.wv.similarity("delivery", "shipping"))
print()
print("Mots sans rapport apparent :")
print("  delivery <-> rude     :", model.wv.similarity("delivery", "rude"))
print("  delivery <-> helpful  :", model.wv.similarity("delivery", "helpful"))

Mots relies (contextes similaires) :
  delivery <-> shipping : 0.9964196

Mots sans rapport apparent :
  delivery <-> rude     : 0.22653265
  delivery <-> helpful  : 0.23043983


In [73]:
# --- BLOC 6 : effet de vector_size (dimensionnalite) ---
# Verifie aussi test_word_vector_has_correct_dimension()
for size in [10, 50, 200]:
    m = train_word2vec(phrases, vector_size=size, window=3, epochs=100, seed=42)
    top1 = most_similar_words(m, "delivery", topn=1)[0]
    vecteur = get_word_vector(m, "delivery")
    print(f"vector_size={size:4} -> voisin : {top1}  |  dimension : {vecteur.shape}")

vector_size=  10 -> voisin : ('shipping', 0.9896082878112793)  |  dimension : (10,)
vector_size=  50 -> voisin : ('shipping', 0.9964196681976318)  |  dimension : (50,)
vector_size= 200 -> voisin : ('shipping', 0.9983150959014893)  |  dimension : (200,)


In [50]:
# --- BLOC 7 : effet de window (taille du contexte) ---
for w in [1, 3, 10]:
    m = train_word2vec(phrases, vector_size=50, window=w, epochs=100, seed=42)
    top1 = most_similar_words(m, "delivery", topn=1)[0]
    print(f"window={w:2} -> voisin le plus proche de 'delivery' : {top1}")

window= 1 -> voisin le plus proche de 'delivery' : ('shipping', 0.9937880635261536)
window= 3 -> voisin le plus proche de 'delivery' : ('shipping', 0.9964196681976318)
window=10 -> voisin le plus proche de 'delivery' : ('shipping', 0.9969074726104736)


## 3. FastText  

In [84]:
from gensim.models import FastText, Word2Vec

phrases_base = [
    "the delivery was fast".split(),
    "the delivery was slow".split(),
    "the shipping was fast".split(),
    "the shipping was slow".split(),
    "the delivery arrived quickly".split(),
    "the shipping arrived quickly".split(),
    "the product quality was great".split(),
    "the product quality was poor".split(),
    "customer service was helpful".split(),
    "customer service was rude".split(),
] * 50

model_w2v = Word2Vec(
    sentences=phrases_base,
    vector_size=50,
    window=3,
    min_count=1,
    sg=1,
    epochs=100,
    seed=42,
)
model_ft = FastText(
    sentences=phrases_base,
    vector_size=50,
    window=3,
    min_count=1,
    sg=1,
    epochs=100,
    seed=42,
)

mot_jamais_vu = "deliveryyy"  # n'existe dans AUCUNE phrase du corpus

print(
    "Le mot",
    repr(mot_jamais_vu),
    "existe dans le corpus ?",
    mot_jamais_vu in " ".join([" ".join(p) for p in phrases_base]),
)
print()

print("--- Word2Vec sur un mot jamais vu ---")
try:
    vecteur = model_w2v.wv[mot_jamais_vu]
    print("Vecteur obtenu :", vecteur[:5])
except KeyError as e:
    print("ECHEC :", e)

print()
print("--- FastText sur le meme mot jamais vu ---")
try:
    vecteur = model_ft.wv[mot_jamais_vu]
    print("Vecteur obtenu (5 premieres valeurs) :", vecteur[:5].round(3))
    print(
        "Voisins les plus proches de ce mot inconnu :",
        model_ft.wv.most_similar(mot_jamais_vu, topn=3),
    )
except KeyError as e:
    print("ECHEC :", e)

Le mot 'deliveryyy' existe dans le corpus ? False

--- Word2Vec sur un mot jamais vu ---
ECHEC : "Key 'deliveryyy' not present"

--- FastText sur le meme mot jamais vu ---
Vecteur obtenu (5 premieres valeurs) : [-0.159 -0.107  0.192 -0.125 -0.086]
Voisins les plus proches de ce mot inconnu : [('delivery', 0.9999302625656128), ('shipping', 0.9974160194396973), ('quickly', 0.990179717540741)]
